# 🤖 RAG Chatbot over a GenAI PDF
**Stack:** LangChain · ChromaDB · Sentence-Transformers · Qwen2.5 (Hugging Face)

Pipeline: `PDF → chunks → embeddings → Chroma → retriever → prompt → LLM → answer`

> Run on Google Colab with a **T4 GPU** (Runtime → Change runtime type). Upload `GenAI_Fundamentals.pdf` to `/content/` first.

## 1. Install

In [ ]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface sentence-transformers transformers accelerate pypdf chromadb

## 2. Settings (change model & temperature here)

In [ ]:
# ---------- EDIT THESE ----------
PDF_PATH       = "/content/GenAI_Fundamentals.pdf"

# Pick one:
#   "Qwen/Qwen2.5-0.5B-Instruct"  -> fastest, CPU ok
#   "Qwen/Qwen2.5-1.5B-Instruct"  -> original model
#   "Qwen/Qwen2.5-3B-Instruct"    -> better quality (GPU)
#   "microsoft/Phi-3.5-mini-instruct"
MODEL_ID       = "Qwen/Qwen2.5-3B-Instruct"

TEMPERATURE    = 0.3    # 0 = deterministic, higher = more creative
TOP_P          = 0.9
MAX_NEW_TOKENS = 250
CHUNK_SIZE     = 750
CHUNK_OVERLAP  = 150
TOP_K          = 4      # chunks retrieved per question
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"

## 3. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

## 4. Load the PDF

In [ ]:
documents = PyPDFLoader(PDF_PATH).load()
print(f"Pages loaded: {len(documents)}")
print(documents[1].page_content[:300])

## 5. Split into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
splits = splitter.split_documents(documents)
print(f"Chunks created: {len(splits)}")

## 6. Embeddings + Chroma vector store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

vector_db = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="genai_rag",          # in-memory; add persist_directory="/content/vector_db" to save
)
print("Vector database created.")

retriever = vector_db.as_retriever(search_kwargs={"k": TOP_K})

### Quick retrieval test

In [ ]:
for d in retriever.invoke("What is RAG?"):
    print("Page", d.metadata["page"] + 1, "->", " ".join(d.page_content.split())[:200], "\n")

## 7. Load the LLM
`do_sample=True` is required, otherwise `temperature` is ignored.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=1.1, return_full_text=False)
if TEMPERATURE > 0:
    gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
else:
    gen_kwargs.update(do_sample=False)

hf_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, **gen_kwargs)
llm = HuggingFacePipeline(pipeline=hf_pipe)

## 8. Build the RAG chain (LCEL)

In [ ]:
def format_docs(docs):
    # remove PDF line-break noise
    return "\n\n".join(" ".join(d.page_content.split()) for d in docs)

SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the question strictly using the provided context. "
    "If the context does not contain the answer, say you don't know. "
    "Keep your answer clear and concise."
)

def build_prompt(inputs):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{inputs['context']}\n\nQuestion: {inputs['question']}"},
    ]
    # uses the model's own chat format, so any model above works
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RunnableLambda(build_prompt)
    | llm
    | StrOutputParser()
)

## 9. Ask questions

In [ ]:
def ask(question, show_sources=True):
    answer = rag_chain.invoke(question).strip()
    print("\n❓", question)
    print("\n✅", answer)
    if show_sources:
        print("\n📚 Sources:")
        for d in retriever.invoke(question):
            print(f"  - Page {d.metadata['page'] + 1}: " + " ".join(d.page_content.split())[:150] + "...")

ask("What is the pdf about?")

In [ ]:
ask("What is Retrieval-Augmented Generation and why is it used?")

In [ ]:
ask("What does the temperature setting do?")

In [ ]:
ask("Explain LoRA in simple words.")

In [ ]:
ask("Who won the FIFA World Cup in 2022?")   # not in the PDF -> should say it does not know

## 10. Chat loop (type `exit` to stop)

In [ ]:
while True:
    q = input("You: ")
    if q.strip().lower() in {"exit", "quit", "q"}:
        break
    ask(q)

## 11. (Optional) Compare the plain LLM vs RAG
Shows why RAG helps: the plain model has no knowledge of your PDF.

In [ ]:
def ask_plain_llm(question):
    msgs = [{"role": "system", "content": "You are a helpful assistant. Keep answers short."},
            {"role": "user", "content": question}]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return hf_pipe(p)[0]["generated_text"].strip()

q = "What chunk size does the document recommend for RAG?"
print("Plain LLM:", ask_plain_llm(q))
print("\nRAG      :", rag_chain.invoke(q).strip())

## 12. (Optional) Run the Streamlit app from Colab
Upload `app.py` + `requirements.txt`, then run:

In [ ]:
!npm install -g localtunnel -q
!curl -s ipv4.icanhazip.com        # copy this IP: it is the tunnel password
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501